# Citadel T1B — Batched multi-scale calculator run (single session)
Preregistered: `docs/citadel/experiments/T1B/PLAN.md`. Five fixed-budget arms (1k→20k updates, same seed/init/data) in ONE session — no dribbles, no TEST peeking (TEST once per arm, all arms reported). T1 (FAIL: loss-learned, exact-flat) is reference arm A0. All logic in `citadel_tpu/`; no secrets.

In [ ]:
# 0. Fresh Citadel checkout + pinned read-only Cymek runtime
import os, subprocess, sys
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = repo
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
if repo not in sys.path: sys.path.insert(0, repo)
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))

In [ ]:
# A. Gates: T1B preflight + T0/T1 context receipts present (T1 is reference arm A0).
import subprocess, os
p1 = subprocess.run(['python','-m','citadel_tpu.calculator_preflight'])
assert p1.returncode == 0, 'T1B PREFLIGHT failed — STOP.'
assert os.path.isfile('docs/citadel/tpu_receipts/TPU_CALCULATOR_CHECKPOINT.json'), 'T1 receipt missing — STOP.'
print('GATES PASS')

In [ ]:
# B. Five fixed-budget arms, one session, in order. Per-arm infra failure is recorded;
# session aborts on the 2nd infra failure. No arm is skipped to reach a larger one.
from citadel_tpu import calculator_train
ARMS = [(1000,'A1k'),(2500,'A2p5k'),(5000,'A5k'),(10000,'A10k'),(20000,'A20k')]
results, infra_failures = {}, 0
t1b = __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()
print('T1B session start:', t1b)
for n, tag in ARMS:
    try:
        r = calculator_train.train(out=f'docs/citadel/tpu_receipts/TPU_SCALE_{tag}.json', ladder=(n,), early_stop=False, ckpt_name=f'tpu_scale_{tag.lower()}.pt')
        results[tag] = (r['status'], r['eval']['trained_test']['accuracy'], r['diagnostics']['memorization_flag'])
    except Exception as e:
        results[tag] = (f'IMPLEMENTATION_FAILURE: {type(e).__name__}', None, None)
        infra_failures += 1
    print(tag, results[tag], flush=True)
    assert infra_failures < 2, '2nd infra failure — ABORT session, report.'
print('T1B arms complete:', results)

In [ ]:
# C. Summary table across arms (all reported; Holm applied before any per-arm PASS claim).
import json
print(f"{'arm':6} {'status':8} {'test_acc':9} {'train_acc':10} {'memorized':10} {'updates':8}")
for n, tag in ARMS:
    try:
        rc = json.load(open(f'docs/citadel/tpu_receipts/TPU_SCALE_{tag}.json'))
        print(f"{tag:6} {rc['status']:8} {rc['eval']['trained_test']['accuracy']:<9.4f} {rc['eval']['trained_train_sample']['accuracy']:<10.4f} {str(rc['diagnostics']['memorization_flag']):<10} {rc['training']['endpoint_updates']:<8}")
    except Exception as e:
        print(tag, 'NO_RECEIPT', repr(e))

In [ ]:
# D. Export all arm receipts + checkpoint binaries of smallest-PASS arm and final executed arm.
import json, glob
from google.colab import files
done = []
for f in sorted(glob.glob('docs/citadel/tpu_receipts/TPU_SCALE_*.json')):
    files.download(f)
    print('exported', f)
    rc = json.load(open(f))
    done.append((f, rc['status'], rc['checkpoint']['path']))
arms = [(f,s,p) for f,s,p in done if s in ('PASS','FAIL')]
passes = [x for x in arms if x[1] == 'PASS']
for label, pick in (('smallest-PASS', passes[0] if passes else None), ('final', arms[-1] if arms else None)):
    if pick is not None:
        files.download(pick[2])
        print('exported', label, pick[2])